# InsightOS Qwen3 0.6B Final Visualization Specialist Training

This notebook refuses blocked approval manifests. It does not use the Qwen smoke adapter.


In [ ]:
!nvidia-smi
import torch
print({"cuda_available": torch.cuda.is_available(), "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, "fp16_supported": torch.cuda.is_available()})


In [ ]:
import hashlib, json, zipfile
BUNDLE = '/kaggle/input/insightos-qwen3-0-6b-final/insightos_qwen3_0_6b_training_ready_bundle.zip'
required_splits = ['train.jsonl', 'validation.jsonl', 'evaluation.jsonl']


In [ ]:
with zipfile.ZipFile(BUNDLE) as z:
    names = z.namelist()
    manifest_members = [n for n in names if n.endswith('training_data/dataset_manifest.json')]
    approval_members = [
        n for n in names
        if n.endswith('approval/final_training_approval_packet.json')
    ]
    if len(manifest_members) != 1:
        raise RuntimeError('Exactly one dataset manifest is required')
    if len(approval_members) != 1:
        raise RuntimeError('Exactly one approval packet is required')
    manifest = json.loads(z.read(manifest_members[0]))
    approval = json.loads(z.read(approval_members[0]))
    if approval['human_approved'] is not True or approval['automatic_approval'] is not False:
        raise RuntimeError('Training refused: explicit human approval is missing')
if manifest['readiness_status'] != 'TRAINING_READY_APPROVED':
    raise RuntimeError('Training refused: bundle readiness is blocked pending explicit approval')
print('Approved dataset verified')


After the approval gate passes, use a verified Unsloth environment, resolve exact model and tokenizer revisions, load the approved bundle, validate the training/evaluation groups, and run QLoRA. The notebook is intentionally fail-closed in its current state.


In [ ]:
!pip -q install unsloth accelerate peft transformers trl datasets bitsandbytes


In [ ]:
import importlib.metadata as metadata
for package in ['torch','transformers','datasets','trl','peft','bitsandbytes','unsloth']:
    try: print(package, metadata.version(package))
    except metadata.PackageNotFoundError: print(package, 'not_installed')


In [ ]:
from datasets import load_dataset
def load_split(name):
    with zipfile.ZipFile(BUNDLE) as z:
        members=[n for n in z.namelist() if n.endswith('/training_data/'+name+'.jsonl')]
        if len(members)!=1:
            raise RuntimeError('Expected exactly one '+name+' split')
        lines=[x for x in z.read(members[0]).decode().splitlines() if x.strip()]
    return load_dataset('json', data_files={'data': lines}, split='data')
train_ds=load_split('train')
validation_ds=load_split('validation')
evaluation_ds=load_split('evaluation')
if len(train_ds)==0 or len(evaluation_ds)==0:
    raise RuntimeError('Readiness requires non-empty approved training and evaluation splits')
print({'train':len(train_ds),'validation':len(validation_ds),'evaluation':len(evaluation_ds)})


In [ ]:
from unsloth import FastLanguageModel
MODEL_ID='unsloth/Qwen3-0.6B-unsloth-bnb-4bit'
model, tokenizer=FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=2048,
    dtype=torch.float16,
    load_in_4bit=True,
)
print({'model':MODEL_ID,'model_config':model.config._name_or_path,'max_position_embeddings':getattr(model.config,'max_position_embeddings',None)})


In [ ]:
model=FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=3407,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
)


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
trainer=SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=validation_ds,
    formatting_func=lambda row: tokenizer.apply_chat_template(
        row['messages'], tokenize=False, add_generation_prompt=False
    ),
    max_seq_length=2048,
    args=TrainingArguments(
        output_dir='/kaggle/working/insightos-qwen3-output',
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        warmup_ratio=0.03,
        max_steps=-1,
        num_train_epochs=3,
        fp16=torch.cuda.is_available(),
        bf16=False,
        logging_steps=10,
        evaluation_strategy='steps',
        eval_steps=25,
        save_strategy='steps',
        save_steps=25,
        save_total_limit=3,
        seed=3407,
        report_to='none',
    ),
)
trainer.train()
trainer.save_model('/kaggle/working/insightos_qwen3_0_6b_adapter')
tokenizer.save_pretrained('/kaggle/working/insightos_qwen3_0_6b_adapter')


In [ ]:
import json
def evaluate_contract(dataset, model, tokenizer):
    if len(dataset)==0:
        raise RuntimeError('Independent evaluation split is empty')
    metrics={'json_parse_rate':0.0,'output_contract_compliance':0.0,
             'supported_visualization_choice':0.0,'field_mapping_correctness':0.0,
             'layout_validity':0.0,'evidence_faithfulness':0.0}
    parsed=0; contract=0; supported=0; mappings=0; layout=0; faithful=0
    for row in dataset:
        messages=row.get('messages') if isinstance(row,dict) else None
        if not messages or len(messages)!=3:
            continue
        prompt=tokenizer.apply_chat_template(
            messages[:2], tokenize=False, add_generation_prompt=True
        )
        inputs=tokenizer(prompt, return_tensors='pt').to(model.device)
        generated_ids=model.generate(
            **inputs, max_new_tokens=768, do_sample=False
        )[0]
        output=tokenizer.decode(
            generated_ids[inputs['input_ids'].shape[1]:],
            skip_special_tokens=True,
        )
        try:
            generated=json.loads(output)
            parsed+=1
        except json.JSONDecodeError:
            continue
        expected=json.loads(messages[2]['content'])
        if generated.get('schema_version')==1:
            contract+=1
        if row.get('task_category')=='visual_reconstruction':
            if generated.get('visualization_id')==expected.get('visualization_id'):
                supported+=1; mappings+=1
        if row.get('task_category')=='dashboard_layout':
            canvas=generated.get('canvas',{})
            if canvas.get('width') and canvas.get('height'):
                layout+=1
        if generated==expected:
            faithful+=1
    total=len(dataset)
    return {
        'json_parse_rate':parsed/total,
        'output_contract_compliance':contract/total,
        'supported_visualization_choice':supported/total,
        'field_mapping_correctness':mappings/total,
        'layout_validity':layout/total,
        'evidence_faithfulness':faithful/total,
    }
import gc, torch
gc.collect(); torch.cuda.empty_cache()
base_model, base_tokenizer=FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=2048,
    dtype=torch.float16,
    load_in_4bit=True,
)
base_metrics=evaluate_contract(evaluation_ds, base_model, base_tokenizer)
tuned_metrics=evaluate_contract(evaluation_ds, model, tokenizer)
json.dump({'base':base_metrics,'tuned':tuned_metrics},open('/kaggle/working/evaluation_metrics.json','w'))
print(json.dumps({'base':base_metrics,'tuned':tuned_metrics},indent=2))
